In [76]:
from pyspark.sql import functions as sf
from pyspark.sql import SparkSession

## SPARK SESSION

In [77]:
#SPARK session creation
def get_spark_session():
  spark=SparkSession.builder.appName("Pokemon").getOrCreate()
  return spark

## EXPLICIT SCHEMA

In [78]:
from pyspark.sql.types import StructType, StructField , IntegerType , StringType , DoubleType

In [79]:
#creating explicit schema
def get_schema():
  #using structType for creating an explicit schema and structField structures the columns in the schema
  schema=StructType([
      StructField("pokemon_id" , IntegerType(), True),
      StructField("pokemon_name" , StringType(), True),
      StructField("height_m", DoubleType(), True),
      StructField("weight_kg", DoubleType(), True),
      StructField("base_experience", IntegerType(), True),
      StructField("hp", IntegerType(), True),
      StructField("attack", IntegerType(), True),
      StructField("defense", IntegerType(), True),
      StructField("special_attack", IntegerType(), True),
      StructField("special_defense", IntegerType(), True),
      StructField("speed", IntegerType(), True)
  ])
  return schema

## CSV DATA EXTRACTION

In [80]:
#read the csv data from the csv file using the spark session object with the schema we explicitily created
def get_csv_data(schema , spark):
  df=spark.read.option("header", True).schema(schema).csv("pokemon.csv")
  return df

## TRANSFORMATION

cleaning the data

In [81]:
#cleaning data
def clean_data(df):
  #this filters out and gives only the non null values of pokemon id
  df=df.filter(sf.col("pokemon_id").isNotNull())
  return df

duplicate records handling

In [82]:
#dealing with duplicates
def remove_duplicates(df):
  #this drops all the duplicate values and returns only non duplicate values
  df=df.dropDuplicates()
  return df

standardizing the data

In [83]:
#standardizing the name data
def standardize_name(df):
  #lower casing the pokemon_name and trimming the extra spaces around
  df=df.withColumn("pokemon_name" , sf.trim(sf.lower(sf.col("pokemon_name"))))
  return df

In [84]:
#standardizing the data
def standardize_num(df):
  #rounding of the numerical decimals to 2 points
  df=df.withColumn("height_m" , sf.round(sf.col("height_m"), 2))
  df=df.withColumn("weight_kg" , sf.round(sf.col("weight_kg"), 2))
  return df

getting the dervied columns

In [85]:
#deriving column 1 ( total_base_stats )
def get_total_base_stats(df):
  #summing up all the stats to create total_base_stat
  df=df.withColumn("total_base_stats" , sf.col("hp")+sf.col("attack")+sf.col("defense")+sf.col("special_attack")+sf.col("special_defense")+sf.col("speed"))
  return df

In [86]:
#deriving column 2 , 3 ( offesive_power and defensive_power )
def get_powers(df):
  #summing up attack and special attack to create offensive power column
  df=df.withColumn("offensive_power", sf.col("attack")+sf.col("special_attack"))
  #summing up defense and special defensive to create defensive power column
  df=df.withColumn("defensive_power", sf.col("defense")+sf.col("special_defense"))
  return df

In [105]:
#deriving column 4 (stat_category)
def get_stat_category(df):
  #if the total_base_stat(sum of stats) is less than 300 , we get Low as stat category
  df=df.withColumn("stat_category" , sf.when(sf.col('total_base_stats')<300 , "Low")

  #if the total base stat is between 300 to 400 then it is medium
  .when(sf.col('total_base_stats')<400 , "Medium")

  #if it is above 400 then it is high
  .otherwise("High"))
  return df

validating by checking null and duplicate's na negative values

In [88]:
#counting the null values in the data frame
def count_null_values(df):
  #select the count of null values
  null_counts=df.select([
      #sum up the 1's and 0's in each column and shows sum of null values in each column
      sf.sum(
          sf.when(
              #when a value in specified column is null then give 1
              sf.col(column).isNull() , 1
              )
          # if it is not null then return 0
          .otherwise(0)
          )
      # name this 1 and 0 value column which has 1 near te null value and 0 near non value with the same column name
      .alias(column)
      # iterate around the columns in my df.columns( al columns in the dataframe )
      for column in df.columns
  ])
  #return the null count
  return null_counts

In [89]:
def get_duplicate_values(df):
  #filter and get the duplicate values
  duplicate_counts=(
  #GROUP the data by pokemon_id
  df.groupBy("pokemon_id")
  # and count how many rows have the same pokemon_id and got grouped
  .count()
  #if there is any pokemon id that is being more than one time filter it and return
  .filter(sf.col("count")>1)
  )
  #return the duplicate count
  return duplicate_counts

In [119]:
def validate_non_negative_stats(df):

    # get records where any numeric value is negative
    invalid_records = df.filter(
        (sf.col("height_m") < 0) |
        (sf.col("weight_kg") < 0) |
        (sf.col("hp") < 0) |
        (sf.col("attack") < 0) |
        (sf.col("defense") < 0) |
        (sf.col("special_attack") < 0) |
        (sf.col("special_defense") < 0) |
        (sf.col("speed") < 0)
    )

    return invalid_records

CALLING ALL THE FUNTIONS HERE

In [90]:
# getting an spark session
spark=get_spark_session()

In [91]:
#get the explicit shcema
schema=get_schema()

In [92]:
schema

StructType([StructField('pokemon_id', IntegerType(), True), StructField('pokemon_name', StringType(), True), StructField('height_m', DoubleType(), True), StructField('weight_kg', DoubleType(), True), StructField('base_experience', IntegerType(), True), StructField('hp', IntegerType(), True), StructField('attack', IntegerType(), True), StructField('defense', IntegerType(), True), StructField('special_attack', IntegerType(), True), StructField('special_defense', IntegerType(), True), StructField('speed', IntegerType(), True)])

In [93]:
#get the csv data
df=get_csv_data(schema , spark)

In [94]:
df.show(5)

+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+
|pokemon_id|pokemon_name|height_m|weight_kg|base_experience| hp|attack|defense|special_attack|special_defense|speed|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+
|         1|   bulbasaur|     0.7|      6.9|           NULL| 45|    49|     49|            65|             65|   45|
|         2|     ivysaur|     1.0|     13.0|           NULL| 60|    62|     63|            80|             80|   60|
|         3|    venusaur|     2.0|    100.0|           NULL| 80|    82|     83|           100|            100|   80|
|         4|  charmander|     0.6|      8.5|           NULL| 39|    52|     43|            60|             50|   65|
|         5|  charmeleon|     1.1|     19.0|           NULL| 58|    64|     58|            80|             65|   80|
+----------+------------+--------+---------+---------------+---+

In [95]:
df.printSchema()

root
 |-- pokemon_id: integer (nullable = true)
 |-- pokemon_name: string (nullable = true)
 |-- height_m: double (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- base_experience: integer (nullable = true)
 |-- hp: integer (nullable = true)
 |-- attack: integer (nullable = true)
 |-- defense: integer (nullable = true)
 |-- special_attack: integer (nullable = true)
 |-- special_defense: integer (nullable = true)
 |-- speed: integer (nullable = true)



In [96]:
df=clean_data(df)

In [97]:
df=remove_duplicates(df)

In [98]:
df=standardize_name(df)

In [99]:
df=standardize_num(df)

In [100]:
df.show(5)

+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+
|pokemon_id|pokemon_name|height_m|weight_kg|base_experience| hp|attack|defense|special_attack|special_defense|speed|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+
|        34|    nidoking|     1.4|     62.0|           NULL| 81|   102|     77|            85|             75|   85|
|       223|    remoraid|     0.6|     12.0|           NULL| 35|    65|     35|            65|             35|   65|
|       369|   relicanth|     1.0|     23.4|           NULL|100|    90|    130|            45|             65|   55|
|       505|     watchog|     1.1|     27.0|           NULL| 60|    85|     69|            60|             69|   77|
|       623|      golurk|     2.8|    330.0|           NULL| 89|   124|     80|            55|             80|   55|
+----------+------------+--------+---------+---------------+---+

In [101]:
df=get_total_base_stats(df)

In [102]:
df=get_powers(df)

In [106]:
df=get_stat_category(df)

In [109]:
df.show(5)

+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
|pokemon_id|pokemon_name|height_m|weight_kg|base_experience| hp|attack|defense|special_attack|special_defense|speed|total_base_stats|offensive_power|defensive_power|stat_category|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
|        34|    nidoking|     1.4|     62.0|           NULL| 81|   102|     77|            85|             75|   85|             505|            187|            152|         High|
|       223|    remoraid|     0.6|     12.0|           NULL| 35|    65|     35|            65|             35|   65|             300|            130|             70|       Medium|
|       369|   relicanth|     1.0|     23.4|           NULL|100|    90|    130|            45|      

In [110]:
df.printSchema()

root
 |-- pokemon_id: integer (nullable = true)
 |-- pokemon_name: string (nullable = true)
 |-- height_m: double (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- base_experience: integer (nullable = true)
 |-- hp: integer (nullable = true)
 |-- attack: integer (nullable = true)
 |-- defense: integer (nullable = true)
 |-- special_attack: integer (nullable = true)
 |-- special_defense: integer (nullable = true)
 |-- speed: integer (nullable = true)
 |-- total_base_stats: integer (nullable = true)
 |-- offensive_power: integer (nullable = true)
 |-- defensive_power: integer (nullable = true)
 |-- stat_category: string (nullable = false)



In [111]:
null_counts=count_null_values(df)

In [112]:
duplicates=get_duplicate_values(df)

In [120]:
invalid_negative_records=validate_non_negative_stats(df)

In [113]:
null_counts.show()

+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
|pokemon_id|pokemon_name|height_m|weight_kg|base_experience| hp|attack|defense|special_attack|special_defense|speed|total_base_stats|offensive_power|defensive_power|stat_category|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
|         0|           0|       0|        0|           1351|  0|     0|      0|             0|              0|    0|               0|              0|              0|            0|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+



In [114]:
duplicates.show()

+----------+-----+
|pokemon_id|count|
+----------+-----+
+----------+-----+



In [121]:
invalid_negative_records.show(5)

+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
|pokemon_id|pokemon_name|height_m|weight_kg|base_experience| hp|attack|defense|special_attack|special_defense|speed|total_base_stats|offensive_power|defensive_power|stat_category|
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+
+----------+------------+--------+---------+---------------+---+------+-------+--------------+---------------+-----+----------------+---------------+---------------+-------------+



## SQL QUERIES

top 5 pokemons with highest total base stats

In [123]:
top_10=df.select("pokemon_id", "pokemon_name" , "total_base_stats").orderBy(sf.col("total_base_stats").desc()).limit(10)

In [124]:
top_10.show()

+----------+-------------------+----------------+
|pokemon_id|       pokemon_name|total_base_stats|
+----------+-------------------+----------------+
|     10190|eternatus-eternamax|            1125|
|     10043|      mewtwo-mega-x|             780|
|     10044|      mewtwo-mega-y|             780|
|     10079|      rayquaza-mega|             780|
|     10301|       zygarde-mega|             778|
|     10077|      kyogre-primal|             770|
|     10078|     groudon-primal|             770|
|     10157|     necrozma-ultra|             754|
|       493|             arceus|             720|
|     10120|   zygarde-complete|             708|
+----------+-------------------+----------------+



## AVERAGE OFFENSIVE AND DEFENSIVE POWER BY STAT CATEGORY

In [125]:
avg_power_by_category=df.groupBy('stat_category').agg(sf.avg('offensive_power').alias('avg_offensive_power'),
sf.avg('defensive_power').alias('avg_defensive_power'))

In [126]:
avg_power_by_category.show()

+-------------+-------------------+-------------------+
|stat_category|avg_offensive_power|avg_defensive_power|
+-------------+-------------------+-------------------+
|         High| 183.87121212121212| 173.17965367965368|
|          Low|  82.66451612903226|  83.56129032258065|
|       Medium| 113.91911764705883| 110.65441176470588|
+-------------+-------------------+-------------------+



## TOP 3 Defensive power pokemons

In [127]:
top_3_defensive=df.select("pokemon_id"  , "pokemon_name" , "defensive_power" , "defense" , "special_defense").orderBy(sf.col("defensive_power")).limit(3)

In [128]:
top_3_defensive.show()

+----------+-------------+---------------+-------+---------------+
|pokemon_id| pokemon_name|defensive_power|defense|special_defense|
+----------+-------------+---------------+-------+---------------+
|       174|    igglybuff|             35|     15|             20|
|       318|     carvanha|             40|     20|             20|
|     10001|deoxys-attack|             40|     20|             20|
+----------+-------------+---------------+-------+---------------+

